# Hospitality Management Analytics - End-to-End Project Requirements

## Project Overview

You are a data engineer working for a large hotel chain. Your task is to build a complete data pipeline that processes raw reservation data, guest profiles, hotel inventory, point-of-sale (POS) transactions, and housekeeping logs from multiple source systems (Front Desk, Website, In-Room Services, Housekeeping) to create business intelligence dashboards for revenue management and operations teams.

The project follows the **Medallion Architecture** pattern:
- **🥉 Bronze**: Raw ingestion layer (provided)
- **🥈 Silver**: Cleaned and conformed data layer (your task)
- **🥇 Gold**: Business intelligence and analytics layer (your task)

**Technology Stack:**
- Databricks (Free Edition - Unity Catalog & Volumes)
- PySpark
- Delta Lake
- Auto Loader (for streaming/batch ingestion)

## 🥉 Bronze Layer (Provided)

The bronze layer contains raw data files in JSON format with **intentional data quality issues**. These are stored in Unity Catalog Volumes.

### Bronze Tables:

1. **`raw_reservations`** (~8,000 rows - Booking Records)
   - `res_id`: Unique reservation identifier
   - `guest_id`: Reference to guest
   - `hotel_id`: Hotel identifier
   - `room_type`: Type of room (Standard, Deluxe, Suite, Presidential)
   - `check_in_date`: Check-in date
   - `check_out_date`: Check-out date
   - `total_price`: Total reservation price
   - `booking_channel`: Booking source (Website, Phone, Walk-in, OTA)
   - `created_at`: Reservation creation timestamp

2. **`raw_guests`** (~5,000 rows - Guest Profile Information)
   - `guest_id`: Unique guest identifier
   - `name`: Guest full name
   - `email`: Guest email address
   - `loyalty_tier`: Loyalty tier (Bronze, Silver, Gold, Platinum)
   - `country`: Guest country of origin
   - `registration_date`: When guest first registered
   - `updated_at`: Last update timestamp

3. **`raw_hotel_inventory`** (~2,000 rows - Physical Room Details)
   - `hotel_id`: Hotel identifier
   - `room_number`: Room number (e.g., "101", "205", "3012")
   - `room_type`: Type of room (Standard, Deluxe, Suite, Presidential)
   - `floor`: Floor number
   - `is_smoking`: Smoking allowed (true/false)
   - `last_renovated`: Date when room was last renovated
   - `max_occupancy`: Maximum number of guests

4. **`raw_pos_transactions`** (~15,000 rows - Restaurant/Bar/Spa Charges)
   - `txn_id`: Unique transaction identifier
   - `res_id`: Reference to reservation (NULL for walk-in guests)
   - `item_name`: Name of item/service purchased
   - `category`: Category (Food, Drink, Service, Spa)
   - `amount`: Transaction amount
   - `timestamp`: Transaction timestamp
   - `guest_id`: Guest identifier (for linking to guest profile)

5. **`raw_housekeeping_logs`** (~12,000 rows - Room Cleaning Status)
   - `log_id`: Unique log identifier
   - `hotel_id`: Hotel identifier
   - `room_number`: Room number
   - `staff_id`: Housekeeping staff identifier
   - `status`: Room status (Clean, Dirty, Maintenance, Out-of-Order)
   - `timestamp`: Status change timestamp

**⚠️ Data Quality Issues to Handle:**
- Missing values (NULL)
- Duplicate records (especially duplicate reservations)
- Invalid date/timestamp formats
- Out-of-range values (negative prices, invalid dates)
- Inconsistent case (mixed uppercase/lowercase in room types, categories)
- Orphaned records (references to non-existent entities)
- **Overbooking**: Two reservations assigned to the same room on the same date
- **Late checkout logic**: POS transactions occurring after check_out_date
- **Dynamic pricing**: Weekend prices are 2x higher than weekday prices (students should analyze this)
- Invalid room numbers (non-existent rooms)
- **Loyalty tier changes**: Guests upgrade tiers over time (SCD Type 2 challenge)

## 🥈 Silver Layer (Your Task)

Create cleaned and conformed tables in the `hospitality_project.silver_schema` schema.

### Silver Tables to Create:

1. **`dim_guests`** (SCD Type 2 - Slowly Changing Dimension)
   - Purpose: Track guest loyalty tier changes over time. A guest who stays in January may be "Bronze" tier, but after multiple stays, they upgrade to "Gold" tier by June. Historical stays must be credited to the correct tier at the time of stay.
   - Include: `guest_id`, `name`, `email`, `loyalty_tier`, `country`, `registration_date`, `valid_from`, `valid_to`, `is_current`
   - Requirements:
     - Clean and standardize guest data from `raw_guests`
     - Handle duplicates (identify same guest with email variations)
     - Standardize loyalty tiers (Bronze, Silver, Gold, Platinum)
     - Validate dates and handle invalid formats
     - Implement SCD Type 2 logic for loyalty tier changes
     - Add `valid_from`, `valid_to`, and `is_current` flags
     - Ensure tier changes are tracked chronologically

2. **`fact_stays_unified`**
   - Purpose: The central "event" table combining reservations with POS transactions to create a "Total Folio" view.
   - Include: `res_id`, `guest_id`, `hotel_id`, `room_type`, `check_in_date`, `check_out_date`, `total_price`, `booking_channel`, `total_pos_amount`, `total_folio_amount`, `pos_item_count`
   - Requirements:
     - Clean reservation data from `raw_reservations`
     - Join with `raw_pos_transactions` to aggregate all charges linked to `res_id`
     - Calculate `total_pos_amount` = sum of all POS transactions for the reservation
     - Calculate `total_folio_amount` = `total_price` + `total_pos_amount` (total wallet share)
     - Count `pos_item_count` = number of POS items purchased
     - Handle NULL `res_id` in POS transactions (walk-in guests - exclude from this fact)
     - Standardize `room_type` and `booking_channel` to proper case
     - Validate dates and handle invalid formats
     - Remove orphaned reservations (guest_id or hotel_id not in dimensions)
     - **Handle late checkout logic**: POS transactions after check_out_date should be linked to previous stay or flagged

3. **`fact_room_availability_daily`**
   - Purpose: A "Calendar" table that explodes reservations into individual daily rows. This is a classic data engineering "date-split" exercise.
   - Include: `date`, `hotel_id`, `room_number`, `room_type`, `is_available`, `res_id`, `guest_id`, `check_in_date`, `check_out_date`
   - Requirements:
     - Join `raw_reservations` with `raw_hotel_inventory` to get room assignments
     - **Date explosion**: For each reservation, create one row for each night the guest stayed
     - Example: If check_in_date = 2024-01-15 and check_out_date = 2024-01-18, create rows for 2024-01-15, 2024-01-16, 2024-01-17
     - Mark `is_available` = false for dates when room is occupied
     - Mark `is_available` = true for dates when room is not occupied
     - Handle date ranges properly (check_out_date is exclusive - guest leaves on that date)
     - **Data quality check**: Flag overbookings (same room_number + date with multiple res_id)
     - Validate room numbers exist in inventory
     - Handle invalid dates and date ranges

**Key Techniques to Use:**
- **Auto Loader** with schema evolution for reading JSON files
- **Rescued data column** to capture malformed records
- **PySpark functions**: `filter()`, `when()`, `regexp_replace()`, `to_date()`, `to_timestamp()`, `upper()`, `lower()`, `trim()`, `explode()`, `sequence()`, `date_add()`
- **Data quality checks**: Validate ranges, formats, and relationships
- **Deduplication**: Use window functions with `row_number()` for reservation deduplication
- **SCD Type 2**: Implement using window functions and date logic for loyalty tier tracking
- **Date explosion**: Use `explode()` or `sequence()` to create daily rows from date ranges
- **Aggregations**: Group by `res_id` to calculate total POS amounts
- **Overbooking detection**: Use window functions to identify duplicate room assignments

## 🥇 Gold Layer (Your Task)

Create business intelligence tables in the `hospitality_project.gold_schema` schema.

### Gold Tables to Create:

1. **`kpi_revpar`** (Revenue Per Available Room)
   - Purpose: The gold standard metric for measuring hotel performance
   - Metric: Total Room Revenue / Total Number of Available Rooms
   - Include: `date`, `hotel_id`, `total_room_revenue`, `total_available_rooms`, `revpar`
   - Requirements:
     - Use `fact_room_availability_daily` to count available rooms per day
     - Use `fact_stays_unified` to calculate total room revenue per day
     - Calculate `revpar` = `total_room_revenue` / `total_available_rooms`
     - Group by date and hotel_id
     - Handle division by zero (when no available rooms)
   - Business Value: Helps revenue managers decide when to raise prices or offer discounts.

2. **`kpi_adr`** (Average Daily Rate)
   - Purpose: Measures the average rate charged per room sold
   - Metric: Total Room Revenue / Number of Rooms Sold
   - Include: `date`, `hotel_id`, `total_room_revenue`, `rooms_sold`, `adr`
   - Requirements:
     - Use `fact_stays_unified` to calculate total revenue and count rooms sold
     - Calculate `adr` = `total_room_revenue` / `rooms_sold`
     - Group by date and hotel_id
     - Handle division by zero
   - Business Value: Tells managers if they are discounting too heavily to fill rooms.

3. **`kpi_ancillary_attachment_rate`**
   - Purpose: Percentage of guests who spent money in Restaurant or Spa in addition to the room
   - Metric: (Guests with POS transactions / Total guests) * 100
   - Include: `date`, `hotel_id`, `total_guests`, `guests_with_ancillary_spend`, `attachment_rate_percentage`
   - Requirements:
     - Use `fact_stays_unified` to identify guests with `pos_item_count` > 0
     - Count total guests and guests with ancillary spend
     - Calculate `attachment_rate_percentage` = (guests_with_ancillary_spend / total_guests) * 100
     - Group by date and hotel_id
     - Filter POS transactions by category (Food, Drink, Service, Spa)
   - Business Value: Helps marketing decide if they should offer "Free Breakfast" packages to increase attachment.

4. **`kpi_housekeeping_turnover_time`**
   - Purpose: Time elapsed between check_out and housekeeping status changing to "Clean"
   - Metric: AVG(time between check_out_date and housekeeping "Clean" status)
   - Include: `date`, `hotel_id`, `total_checkouts`, `avg_turnover_time_minutes`, `median_turnover_time_minutes`
   - Requirements:
     - Join `fact_stays_unified` (check_out_date) with `raw_housekeeping_logs` (status = 'Clean')
     - Match by `hotel_id` and `room_number` (need to infer room_number from reservation)
     - Calculate time difference in minutes between check_out and "Clean" status
     - Filter for same-day or next-day cleanings
     - Calculate average and median turnover time
     - Group by date and hotel_id
   - Business Value: Helps operations optimize housekeeping schedules and room readiness.

5. **`kpi_weekend_vs_weekday_revenue`** (Bonus)
   - Purpose: Analyze revenue trends between weekends and weekdays
   - Metric: Compare ADR and RevPAR for weekends vs weekdays
   - Include: `date`, `hotel_id`, `is_weekend`, `adr`, `revpar`, `occupancy_rate`
   - Requirements:
     - Identify weekends (Saturday, Sunday) vs weekdays
     - Aggregate ADR and RevPAR by weekend/weekday
     - Calculate occupancy rate = rooms_sold / total_available_rooms
     - Group by date, hotel_id, and is_weekend flag
   - Business Value: Validates dynamic pricing strategy (weekend prices should be 2x higher).

**Key Techniques to Use:**
- **Aggregations**: `groupBy()`, `agg()`, `sum()`, `avg()`, `max()`, `min()`, `count()`
- **Window functions**: For time-based calculations (turnover time, date comparisons)
- **Joins**: Combine fact and dimension tables
- **Date/time functions**: Extract date, calculate time differences, date arithmetic, day of week
- **Conditional logic**: `when()`, `case when` for flagging weekends, calculating rates
- **Division by zero handling**: Use `when()` to handle cases where denominator is zero

## Implementation Guidelines

### Step 1: Environment Setup
1. Run the `01_hospitality_data_setup.ipynb` notebook to create bronze data
2. Create schemas for silver and gold layers:
   ```sql
   CREATE SCHEMA IF NOT EXISTS hospitality_project.silver_schema;
   CREATE SCHEMA IF NOT EXISTS hospitality_project.gold_schema;
   ```

### Step 2: Bronze to Silver Pipeline
1. Use **Auto Loader** to read JSON files from the volume
2. Enable **schema evolution** and **rescued data column**
3. Clean and transform each bronze table
4. Handle guest deduplication (same email, similar names)
5. Implement SCD Type 2 for `dim_guests` (loyalty tier changes)
6. Create `fact_stays_unified` by joining reservations with POS transactions
7. Create `fact_room_availability_daily` using date explosion technique
8. Write to Delta tables in silver schema

### Step 3: Silver to Gold Pipeline
1. Read from silver tables
2. Perform aggregations and calculations for each KPI
3. Handle overbooking detection in room availability
4. Calculate time-based metrics (turnover time)
5. Write to Delta tables in gold schema

### Step 4: Data Quality Checks
1. Validate record counts
2. Check for NULLs in critical fields
3. Verify relationships (foreign keys)
4. Validate calculated metrics (ranges, formats)
5. Check for duplicate reservations resolved correctly
6. **Detect overbookings**: Same room + date with multiple reservations
7. **Validate late checkout logic**: POS transactions after check_out_date

### Code Structure Example:
```python
# Read bronze data with Auto Loader
bronze_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/path/to/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load("/Volumes/hospitality_project/bronze_schema/raw/reservations"))

# Clean and transform
silver_df = (bronze_df
    .filter(col("res_id").isNotNull())
    .withColumn("room_type", trim(upper(col("room_type"))))
    .withColumn("check_in_date", to_date(col("check_in_date")))
    .dropDuplicates(["res_id"]))

# Write to Delta
(silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/path/to/checkpoint")
    .table("hospitality_project.silver_schema.dim_guests"))
```

## Deliverables

1. **Silver Layer Tables** (3 tables):
   - `dim_guests` (SCD Type 2)
   - `fact_stays_unified`
   - `fact_room_availability_daily`

2. **Gold Layer Tables** (4-5 tables):
   - `kpi_revpar`
   - `kpi_adr`
   - `kpi_ancillary_attachment_rate`
   - `kpi_housekeeping_turnover_time`
   - `kpi_weekend_vs_weekday_revenue` (bonus)

3. **Documentation**:
   - Explain your data cleaning approach
   - Document how you handled guest deduplication
   - Describe your SCD Type 2 implementation for loyalty tiers
   - Explain how you implemented date explosion for room availability
   - Document how you handled overbooking detection
   - Explain late checkout logic (POS transactions after check_out_date)
   - Describe how you calculated total folio amounts
   - Document any assumptions made
   - Describe how you handled data quality issues
   - Explain your KPI calculation logic

4. **Data Quality Report**:
   - Number of records processed
   - Number of records filtered/dropped
   - Number of rescued records
   - Number of duplicate reservations identified and resolved
   - Number of overbookings detected
   - Number of late checkout POS transactions
   - Data quality metrics

## Evaluation Criteria

1. **Data Quality** (40%):
   - Proper handling of missing values
   - Duplicate removal (especially reservation deduplication)
   - Data type conversions
   - Validation of ranges and formats
   - Use of rescued data column
   - Overbooking detection and handling
   - Late checkout logic handling

2. **Data Modeling** (30%):
   - Correct SCD Type 2 implementation for loyalty tiers
   - Proper dimension and fact table design
   - Appropriate joins and relationships
   - Guest deduplication logic
   - Date explosion implementation for room availability
   - Total folio calculation (reservation + POS aggregation)

3. **Business Logic** (20%):
   - Correct KPI calculations (RevPAR, ADR, Attachment Rate, Turnover Time)
   - Accurate aggregations
   - Proper date/time handling
   - Weekend vs weekday analysis
   - Dynamic pricing validation

4. **Code Quality** (10%):
   - Clean, readable code
   - Proper use of PySpark functions
   - Efficient transformations
   - Comments and documentation

## Tips and Best Practices

1. **Start with Bronze to Silver**: Focus on cleaning one table at a time, starting with `dim_guests`
2. **Guest Deduplication**: Use email + name similarity to identify duplicate guests
3. **SCD Type 2**: Track loyalty tier changes over time using window functions and date logic
4. **Date Explosion**: Use `explode()` with `sequence()` or date arithmetic to create daily rows from check_in/check_out ranges
5. **Overbooking Detection**: Use window functions to identify rooms with multiple reservations on the same date
6. **Total Folio**: Aggregate POS transactions by `res_id` and join back to reservations
7. **Late Checkout Logic**: Identify POS transactions with timestamp after check_out_date and decide how to handle (link to previous stay or flag)
8. **Test Incrementally**: Verify each transformation step
9. **Handle Edge Cases**: NULL values, empty strings, extreme values, invalid dates, date ranges where check_out < check_in
10. **Use Rescued Data**: Check `_rescued_data` column for malformed records
11. **Optimize Joins**: Use broadcast joins for small dimension tables
12. **Partition Strategically**: Partition fact tables by date for better performance
13. **Document Assumptions**: Note any business logic decisions (e.g., how to handle late checkout POS, overbooking resolution)
14. **Validate Results**: Check record counts and sample data after each step
15. **Weekend Detection**: Use `dayofweek()` function to identify weekends (Saturday=6, Sunday=0)

## Resources

- [Databricks Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
- [Delta Lake Documentation](https://docs.delta.io/)
- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)
- [SCD Type 2 Implementation](https://www.databricks.com/blog/2022/11/30/implementing-slowly-changing-dimensions-scd-type-2-using-delta-lake.html)

---